# Lab 3 — TASK 2.4: Xác minh chữ ký chứng chỉ X.509 bằng RSA


## 1. Phân tích & Tư duy

Xác minh chữ ký do **Amazon RSA 2048 M04** tạo trên chứng chỉ
`www.amazon.com`. Đầu vào là các tệp đã trích xuất bằng OpenSSL ở bước 1–4.

| Dữ liệu | Tệp |
|---|---|
| Chứng chỉ máy chủ | `c0.pem` |
| Chứng chỉ CA phát hành trực tiếp | `c1.pem` |
| Khóa công khai CA | `ca_public.pem` |
| Modulus RSA của CA | `modulus.txt` |
| Chữ ký dạng hex | `signature.hex` |
| Phần thân DER được ký | `c0_body.bin` |
| SHA-256 đã tính bằng PowerShell | `c0_body.sha256.txt` |

`chain.txt`, `c0_details.txt`, `signature.txt` là dữ liệu hỗ trợ thao tác thủ
công; phép tính chỉ cần các tệp trong bảng.

Với khóa công khai $(n,e)$, chuyển chữ ký thành số nguyên big-endian $s$:

$$EM_{recovered}=\operatorname{I2OSP}(s^e\bmod n,k),
\qquad k=\left\lceil\frac{\operatorname{bitlen}(n)}8\right\rceil.$$

Đây không phải phép khôi phục trực tiếp văn bản gốc hay riêng hash. Đối với
SHA-256 với RSA PKCS#1 v1.5, khối đúng phải có dạng:

$$EM_{expected}=\texttt{00 01}\;||\;\texttt{FF…FF}\;||\;\texttt{00}
\;||\;DigestInfoPrefix_{SHA256}\;||\;SHA256(TBS).$$

Tiền tố DER của SHA-256 là `3031300d060960864801650304020105000420`, dài
19 byte. Hash dài 32 byte. Với RSA 2048 bit, $k=256$, padding dài
$256-3-19-32=202$ byte, và phải có ít nhất 8 byte `FF`.

**Điểm cần tránh:** không băm toàn bộ tệp PEM, không dùng khóa máy chủ thay
cho khóa CA, không chỉ so 32 byte cuối mà bỏ qua padding. Các byte TBS phải
bao gồm header DER của chính `TBSCertificate`.

`cryptography` được dùng để phân tích chứng chỉ/khóa và đối chiếu cuối cùng.
Phép xác minh chính được tự cài đặt bằng `pow(s, e, n)`.

## 2. Thiết kế Mã giả

```text
ĐỌC chứng chỉ c0, c1 và khóa CA
KIỂM TRA thuật toán của c0 là SHA-256/RSA PKCS#1 v1.5
KIỂM TRA issuer(c0) = subject(c1)
LẤY n, e từ khóa CA và đối chiếu modulus.txt, khóa trong c1
ĐỌC chữ ký S, TBS và hash đã lưu
ĐỐI CHIẾU S, TBS với các trường trong c0
TÍNH SHA256(TBS), đối chiếu hash đã lưu

k ← ceil(bit_length(n) / 8)
TỪ CHỐI nếu chữ ký không dài k byte hoặc s >= n
EM_recovered ← pow(s, e, n), biểu diễn đúng k byte
EM_expected ← 00 01 || FF…FF || 00 || DER_SHA256 || SHA256(TBS)
KẾT LUẬN chữ ký đúng khi toàn bộ hai khối bằng nhau
ĐỐI CHIẾU với cryptography và kiểm thử đầu vào bị thay đổi
```

## 3. Mã nguồn Python


### 1 — Cài thư viện

Chỉ cần chạy nếu kernel chưa có `cryptography`. Thư viện này phân tích X.509
và khóa công khai; không thay thế phép tính RSA thủ công ở Cell 5.


In [ ]:
%pip install cryptography


### 2 — Import và đọc tệp văn bản do PowerShell tạo

Đặt notebook cùng thư mục với các tệp đầu vào rồi chạy từ trên xuống.
`DATA_DIR = Path(".")` dùng thư mục làm việc hiện tại, không chứa đường dẫn
cứng theo tên tài khoản. Nếu cần, thay bằng một đường dẫn tương đối phù hợp.

`modulus.txt` bạn gửi có BOM UTF-16; các tệp hex/hash là ASCII hoặc UTF-8.
Hàm dưới nhận diện BOM để đọc đúng cả hai dạng, không làm thay đổi tệp gốc.


In [ ]:
import hashlib
import hmac
from pathlib import Path

from cryptography import x509
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.x509.oid import SignatureAlgorithmOID


DATA_DIR = Path(".")
SHA256_PREFIX = bytes.fromhex("3031300d060960864801650304020105000420")


def read_text_file(path: Path) -> str:
    """Đọc văn bản UTF-16 có BOM hoặc UTF-8/ASCII, bỏ BOM nếu có."""
    raw = path.read_bytes()
    has_utf16_bom = raw.startswith((b"\xff\xfe", b"\xfe\xff"))
    encoding = "utf-16" if has_utf16_bom else "utf-8-sig"
    return raw.decode(encoding).strip()


### 3 — Đọc chứng chỉ và lấy khóa RSA của CA

Cell này kiểm tra thuật toán chữ ký của chứng chỉ máy chủ, quan hệ tên
issuer/subject và sự khớp nhau giữa khóa trong `ca_public.pem` với khóa
trong `c1.pem`. Sau đó lấy `(n, e)` từ khóa CA và đối chiếu `modulus.txt`.

Tên issuer/subject trùng nhau chưa đủ xác minh chữ ký. Các kiểm tra này giúp
phát hiện chọn nhầm tệp trước khi thực hiện phép toán RSA.


In [ ]:
leaf = x509.load_pem_x509_certificate((DATA_DIR / "c0.pem").read_bytes())
issuer = x509.load_pem_x509_certificate((DATA_DIR / "c1.pem").read_bytes())
ca_key = serialization.load_pem_public_key(
    (DATA_DIR / "ca_public.pem").read_bytes()
)
issuer_key = issuer.public_key()

if leaf.signature_algorithm_oid != SignatureAlgorithmOID.RSA_WITH_SHA256:
    raise ValueError("Cần chữ ký sha256WithRSAEncryption trên c0.pem.")
if not isinstance(ca_key, rsa.RSAPublicKey):
    raise TypeError("ca_public.pem phải chứa khóa công khai RSA.")
if not isinstance(issuer_key, rsa.RSAPublicKey):
    raise TypeError("c1.pem phải chứa khóa công khai RSA.")
if leaf.issuer != issuer.subject:
    raise ValueError("Issuer của c0 không khớp subject của c1.")
if ca_key.public_numbers() != issuer_key.public_numbers():
    raise ValueError("Khóa trong ca_public.pem không khớp c1.pem.")

key_numbers = ca_key.public_numbers()
n, e = key_numbers.n, key_numbers.e
modulus_text = read_text_file(DATA_DIR / "modulus.txt")
label, separator, modulus_hex = modulus_text.partition("=")
if label.strip() != "Modulus" or not separator:
    raise ValueError("modulus.txt phải có dạng Modulus=<giá trị hex>.")
if int(modulus_hex, 16) != n:
    raise ValueError("Modulus đã trích xuất không khớp khóa CA.")

print("Máy chủ:", leaf.subject.rfc4514_string())
print("CA:", issuer.subject.rfc4514_string())
print("Kích thước RSA:", ca_key.key_size, "bit")
print("e =", e)
print("n (hex) =", format(n, "x"))


Máy chủ: CN=www.amazon.com
CA: CN=Amazon RSA 2048 M04,O=Amazon,C=US
Kích thước RSA: 2048 bit
e = 65537
n (hex) = cde9551e809503a9e317bbefa15d78f5f27a3800dfc998c75a2ef7f8eba55e927e7fbfa2a04a830858e9e3ec4071d06b258125d067bf85b3dfadea448e17521f6cc01149d754e76772ea38cbc7996ab6fa5a00d103ed39f5dd2767c704bac1ae782d27f181d29e68940558ad2660bd018fd7a45ff1d76d372c44121626dc7ba198d8bebd2780ab5b4b6093c0bd1097a274e959d483c58ea1bdd9b196a4d88bcd80b415d27e9190e7c1c7334870d02e9a41a92086bc168b97c6b252ceab3a6298d308b13edca9ad06271e897caa89aa641036994eda1f27f59b05260068c287f4d69a5da0b01683a71a30e334a26fd520b29a6b7d05f56febf108da87b96d1959


### 4 — Đọc chữ ký, TBS và kiểm tra hash đã lưu

`signature.hex` được chuyển từ hex thành các byte chữ ký. `c0_body.bin`
được đọc ở chế độ nhị phân để giữ nguyên các byte DER. Chương trình đối
chiếu cả hai tệp với các trường được phân tích từ `c0.pem` nhằm kiểm tra
các bước trích xuất thủ công.

Tiếp theo, tính SHA-256(TBS) bằng Python và so với kết quả PowerShell.
Hash trùng nhau mới chỉ chứng minh hai phép băm nhất quán, chưa xác minh
chữ ký RSA. Nếu dữ liệu trích xuất sai, cell báo lỗi để sửa trước khi tiếp tục.


In [ ]:
signature_hex = read_text_file(DATA_DIR / "signature.hex")
signature = bytes.fromhex(signature_hex)
tbs = (DATA_DIR / "c0_body.bin").read_bytes()
saved_hash = read_text_file(DATA_DIR / "c0_body.sha256.txt").lower()
digest = hashlib.sha256(tbs).digest()

if signature != leaf.signature:
    raise ValueError("signature.hex không khớp chữ ký trong c0.pem.")
if tbs != leaf.tbs_certificate_bytes:
    raise ValueError("c0_body.bin không khớp DER của TBSCertificate.")
if not hmac.compare_digest(digest.hex(), saved_hash):
    raise ValueError("SHA-256 tính lại không khớp hash đã lưu.")

print("Chữ ký trích xuất khớp chứng chỉ: True")
print("TBS trích xuất khớp chứng chỉ: True")
print("Độ dài chữ ký:", len(signature), "byte")
print("Độ dài TBS:", len(tbs), "byte")
print("SHA-256(TBS):", digest.hex())
print("Hash khớp kết quả PowerShell: True")


Chữ ký trích xuất khớp chứng chỉ: True
TBS trích xuất khớp chứng chỉ: True
Độ dài chữ ký: 256 byte
Độ dài TBS: 1711 byte
SHA-256(TBS): 2086440c0eaa3387cea8264d8fb5d54238aaa590d0641c81b3761be0399452aa
Hash khớp kết quả PowerShell: True


### 5 — Tự xác minh bằng công thức RSA và PKCS#1 v1.5

`recover_rsa_block` kiểm tra độ dài chữ ký và miền `0 <= s < n`, rồi
tính `pow(s, e, n)`. Kết quả được chuyển thành **đúng k byte** để giữ
byte `00` ở đầu. `encode_sha256` dựng khối mong đợi gồm header, padding
`FF`, `DigestInfo` và hash SHA-256.

`verify_rsa_sha256` so sánh toàn bộ hai khối. Tách các bước thành hàm giúp
kiểm thử mà không phải đọc lại tệp. `(n, e)` ở đây là các tham số của khóa
RSA hợp lệ đã phân tích ở Cell 3; các hàm này không kiểm định việc sinh khóa.


In [ ]:
def recover_rsa_block(sig: bytes, modulus: int, exponent: int) -> bytes:
    """Khôi phục khối RSA bằng s^e mod n và giữ đủ độ dài modulus."""
    block_size = (modulus.bit_length() + 7) // 8
    if len(sig) != block_size:
        raise ValueError("Độ dài chữ ký không đúng kích thước khóa RSA.")

    s = int.from_bytes(sig, byteorder="big")
    if s >= modulus:
        raise ValueError("Đại diện chữ ký phải nhỏ hơn modulus n.")

    m = pow(s, exponent, modulus)
    return m.to_bytes(block_size, byteorder="big")


def encode_sha256(data: bytes, block_size: int) -> bytes:
    """Tạo toàn bộ khối EMSA-PKCS1-v1_5 tương ứng SHA-256(data)."""
    digest_info = SHA256_PREFIX + hashlib.sha256(data).digest()
    padding_size = block_size - len(digest_info) - 3
    if padding_size < 8:
        raise ValueError("Khóa quá ngắn cho SHA-256 với PKCS#1 v1.5.")
    return b"\x00\x01" + b"\xff" * padding_size + b"\x00" + digest_info


def verify_rsa_sha256(
    data: bytes, sig: bytes, modulus: int, exponent: int
) -> bool:
    """Xác minh thủ công SHA-256/RSA PKCS#1 v1.5 với khóa RSA đã đọc."""
    try:
        recovered = recover_rsa_block(sig, modulus, exponent)
        expected = encode_sha256(data, len(recovered))
    except ValueError:
        return False
    return hmac.compare_digest(recovered, expected)


em_recovered = recover_rsa_block(signature, n, e)
em_expected = encode_sha256(tbs, len(em_recovered))
manual_valid = verify_rsa_sha256(tbs, signature, n, e)

print("Độ dài khối EM:", len(em_recovered), "byte")
print("EM khôi phục (hex):", em_recovered.hex())
print("EM mong đợi  (hex):", em_expected.hex())
print("CHỮ KÝ RSA HỢP LỆ:", manual_valid)


Độ dài khối EM: 256 byte
EM khôi phục (hex): 0001ffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff003031300d0609608648016503040201050004202086440c0eaa3387cea8264d8fb5d54238aaa590d0641c81b3761be0399452aa
EM mong đợi  (hex): 0001ffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffffff003031300d0609

## 4. Minh họa & Chứng minh

### 6 — Đối chiếu với cryptography và thử dữ liệu bị sửa

Phép kiểm tra chính đã hoàn thành ở Cell 5. Cell này dùng API `.verify()`
để đối chiếu độc lập: thư viện không trả boolean mà trả `None` khi đúng và
ném `InvalidSignature` khi sai. Hàm phụ chuyển hành vi đó thành boolean.

Năm ca kiểm thử gồm dữ liệu gốc, sửa một byte TBS, sửa một byte chữ ký,
cắt chữ ký và dùng nhầm khóa công khai của máy chủ. Mọi ca đều được đối
chiếu với thư viện. Việc sửa TBS chỉ áp dụng lên một bản sao trong RAM,
không ghi đè các tệp của bạn.


In [ ]:
def reference_verify(data: bytes, sig: bytes, public_key) -> bool:
    """Đối chiếu bằng thư viện; không dùng cho phép xác minh thủ công."""
    try:
        public_key.verify(sig, data, padding.PKCS1v15(), hashes.SHA256())
    except InvalidSignature:
        return False
    return True


library_valid = reference_verify(tbs, signature, ca_key)
assert manual_valid == library_valid, "Hai cách xác minh không khớp."
print("Kết quả thủ công:", manual_valid)
print("Kết quả thư viện:", library_valid)

changed_tbs = tbs[:-1] + bytes([tbs[-1] ^ 1])
changed_sig = signature[:-1] + bytes([signature[-1] ^ 1])
cases = [
    ("Dữ liệu gốc", tbs, signature, ca_key, True),
    ("Sửa một byte TBS", changed_tbs, signature, ca_key, False),
    ("Sửa một byte chữ ký", tbs, changed_sig, ca_key, False),
    ("Cắt chữ ký", tbs, signature[:-1], ca_key, False),
    ("Dùng khóa máy chủ", tbs, signature, leaf.public_key(), False),
]

for name, data, sig, key, expected in cases:
    numbers = key.public_numbers()
    actual = verify_rsa_sha256(data, sig, numbers.n, numbers.e)
    reference = reference_verify(data, sig, key)
    assert actual == reference == expected, name
    print(f"PASS | {name:<22} | hợp lệ = {actual}")


Kết quả thủ công: True
Kết quả thư viện: True
PASS | Dữ liệu gốc            | hợp lệ = True
PASS | Sửa một byte TBS       | hợp lệ = False
PASS | Sửa một byte chữ ký    | hợp lệ = False
PASS | Cắt chữ ký             | hợp lệ = False
PASS | Dùng khóa máy chủ      | hợp lệ = False


### Kết quả thực nghiệm trên các tệp đã gửi

- Máy chủ: `www.amazon.com`.
- CA trực tiếp: `Amazon RSA 2048 M04`.
- Thuật toán chữ ký: SHA-256 với RSA PKCS#1 v1.5.
- Khóa CA: RSA 2048 bit; `e = 65537`.
- Chữ ký: **256 byte**; TBS: **1711 byte**.
- SHA-256(TBS):
  `2086440c0eaa3387cea8264d8fb5d54238aaa590d0641c81b3761be0399452aa`.
- Toàn bộ khối RSA khôi phục trùng khối mã hóa mong đợi: **True**.
- Kết quả thư viện: **True**; **5/5** ca kiểm thử đạt.

Chữ ký trên chứng chỉ máy chủ hợp lệ với khóa công khai của CA được cung
cấp. Các ca thay đổi dữ liệu, chữ ký hoặc dùng sai khóa đều bị từ chối.
Kết quả này chưa xác thực đầy đủ chuỗi tin cậy TLS, thời hạn, tên miền,
quyền phát hành chứng chỉ hay trạng thái thu hồi.

Tài liệu tham khảo:

- [RFC 8017 — RSA PKCS#1, mục 8.2.2 và 9.2](https://www.rfc-editor.org/rfc/rfc8017.html)
- [RFC 5280 — Cấu trúc chứng chỉ X.509](https://www.rfc-editor.org/rfc/rfc5280.html)
- [Cryptography — X.509 reference](https://cryptography.io/en/latest/x509/reference/)
